In [ ]:
!pip install datasets transformers accelerate
import torch
import os
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm


In [ ]:
MODEL_NAME = "EleutherAI/pythia-2.8b-deduped"
LAYER_IDX = 24
TARGET_TOKENS = 1_000_000
BATCH_SIZE = 4
MAX_SEQ_LEN = 512

SAVE_DIR = "/content/drive/MyDrive/SAE_Works/Pythia-2.8B/pile_layer24"
os.makedirs(SAVE_DIR, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/5.68G [00:00<?, ?B/s]

GPTNeoXForCausalLM(
  (gpt_neox): GPTNeoXModel(
    (embed_in): Embedding(50304, 2560)
    (emb_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-31): 32 x GPTNeoXLayer(
        (input_layernorm): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
        (post_attention_layernorm): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
        (post_attention_dropout): Dropout(p=0.0, inplace=False)
        (post_mlp_dropout): Dropout(p=0.0, inplace=False)
        (attention): GPTNeoXAttention(
          (query_key_value): Linear(in_features=2560, out_features=7680, bias=True)
          (dense): Linear(in_features=2560, out_features=2560, bias=True)
        )
        (mlp): GPTNeoXMLP(
          (dense_h_to_4h): Linear(in_features=2560, out_features=10240, bias=True)
          (dense_4h_to_h): Linear(in_features=10240, out_features=2560, bias=True)
          (act): GELUActivation()
        )
      )
    )
    (final_layer_norm): LayerNorm((2560,), eps=1e-05

In [ ]:
activation_buffer = []

def activation_hook(module, input, output):
    # output: [batch, seq, d_model]
    activation_buffer.append(output.detach().cpu())


In [ ]:
hook_handle = model.gpt_neox.layers[LAYER_IDX].mlp.register_forward_hook(activation_hook)


In [ ]:
pile = load_dataset(
    "EleutherAI/the_pile_deduplicated",
    split="train",
    streaming=True
)


Resolving data files:   0%|          | 0/1650 [00:00<?, ?it/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

In [ ]:
token_count = 0
shard_id = 0
shard_acts = []

progress = tqdm(total=TARGET_TOKENS, desc="Collecting tokens")

for sample in pile:
    if token_count >= TARGET_TOKENS:
        break

    text = sample["text"]
    if not isinstance(text, str) or len(text.strip()) == 0:
        continue

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN
    ).to(model.device)

    with torch.no_grad():
        _ = model(**inputs)

    if len(activation_buffer) == 0:
        continue

    acts = activation_buffer.pop()  # [1, seq, d_model]
    acts = acts.squeeze(0)           # [seq, d_model]

    shard_acts.append(acts)
    token_count += acts.shape[0]
    progress.update(acts.shape[0])

    # Save shard every ~50k tokens
    if token_count // 50_000 > shard_id:
        shard_tensor = torch.cat(shard_acts, dim=0)
        save_path = os.path.join(
            SAVE_DIR, f"pile_layer24_shard_{shard_id:03d}.pt"
        )
        torch.save(shard_tensor, save_path)

        print(f"Saved shard {shard_id} → {shard_tensor.shape}")

        shard_acts.clear()
        shard_id += 1


Saved shard 0 → torch.Size([50111, 2560])
Saved shard 1 → torch.Size([49962, 2560])
Saved shard 2 → torch.Size([50209, 2560])
Saved shard 3 → torch.Size([49736, 2560])
Saved shard 4 → torch.Size([50357, 2560])
Saved shard 5 → torch.Size([50004, 2560])
Saved shard 6 → torch.Size([50058, 2560])
Saved shard 7 → torch.Size([49650, 2560])
Saved shard 8 → torch.Size([50231, 2560])
Saved shard 9 → torch.Size([50075, 2560])
Saved shard 10 → torch.Size([49782, 2560])
Saved shard 11 → torch.Size([50168, 2560])
Saved shard 12 → torch.Size([49928, 2560])
Saved shard 13 → torch.Size([49736, 2560])
Saved shard 14 → torch.Size([50269, 2560])
Saved shard 15 → torch.Size([49819, 2560])
Saved shard 16 → torch.Size([49962, 2560])
Saved shard 17 → torch.Size([50259, 2560])
Saved shard 18 → torch.Size([49781, 2560])
Saved shard 19 → torch.Size([50382, 2560])


In [ ]:
# Save remaining activations
if len(shard_acts) > 0:
    shard_tensor = torch.cat(shard_acts, dim=0)
    save_path = os.path.join(
        SAVE_DIR, f"pile_layer24_shard_{shard_id:03d}.pt"
    )
    torch.save(shard_tensor, save_path)
    print(f"Saved final shard {shard_id} → {shard_tensor.shape}")

hook_handle.remove()
print(f"✓ Done. Collected ~{token_count:,} tokens.")


✓ Done. Collected ~1,000,479 tokens.


In [ ]:
SAVE_DIR = "/content/Jimmy/MyDrive/SAE_Works/Pythia-2.8B/pile_layer24"

In [ ]:
import os
import shutil

source_dir = SAVE_DIR # This variable was defined earlier
target_dir = '/content/drive/MyDrive/SAE_Works_Pythia-2.8B/layer 24'

# Create the target directory if it doesn't exist
os.makedirs(target_dir, exist_ok=True)

print(f"Moving files from {source_dir} to {target_dir}...")

moved_count = 0
for filename in os.listdir(source_dir):
    if filename.startswith('pile_layer24_shard_') and filename.endswith('.pt'):
        try:
            # Extract shard ID
            shard_id_str = filename.split('_')[-1].split('.')[0]
            shard_id = int(shard_id_str)

            # Construct new filename
            new_filename = f"pythia-2.8B-deduped_layer_24_shard{shard_id:03d}.pt"

            source_path = os.path.join(source_dir, filename)
            target_path = os.path.join(target_dir, new_filename)

            shutil.move(source_path, target_path)
            print(f"Moved and renamed: {filename} -> {new_filename}")
            moved_count += 1
        except Exception as e:
            print(f"Error moving file {filename}: {e}")

print(f"✓ Done. Moved {moved_count} shard files.")

# Optionally, remove the old directory if it's empty after moving
if not os.listdir(source_dir):
    os.rmdir(source_dir)
    print(f"Removed empty source directory: {source_dir}")
else:
    print(f"Source directory {source_dir} is not empty after move. Remaining files: {os.listdir(source_dir)}")

Moving files from /content/Jimmy/MyDrive/SAE_Works/Pythia-2.8B/pile_layer24 to /content/drive/MyDrive/SAE_Works_Pythia-2.8B/layer 24...
Moved and renamed: pile_layer24_shard_011.pt -> pythia-2.8B-deduped_layer_24_shard011.pt
Moved and renamed: pile_layer24_shard_000.pt -> pythia-2.8B-deduped_layer_24_shard000.pt
Moved and renamed: pile_layer24_shard_012.pt -> pythia-2.8B-deduped_layer_24_shard012.pt
Moved and renamed: pile_layer24_shard_019.pt -> pythia-2.8B-deduped_layer_24_shard019.pt
Moved and renamed: pile_layer24_shard_017.pt -> pythia-2.8B-deduped_layer_24_shard017.pt
Moved and renamed: pile_layer24_shard_014.pt -> pythia-2.8B-deduped_layer_24_shard014.pt
Moved and renamed: pile_layer24_shard_018.pt -> pythia-2.8B-deduped_layer_24_shard018.pt
Moved and renamed: pile_layer24_shard_009.pt -> pythia-2.8B-deduped_layer_24_shard009.pt
Moved and renamed: pile_layer24_shard_002.pt -> pythia-2.8B-deduped_layer_24_shard002.pt
Moved and renamed: pile_layer24_shard_006.pt -> pythia-2.8B-ded

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

ValueError: Mountpoint must not already contain files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
